In [113]:
# ==============================================================================
# CONNECT-TEL CHURN PREDICTION - XGBOOST UPGRADE
# ==============================================================================

import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb  # <-- NEW IMPORT

print("1. Loading Data...")
df = pd.read_csv('../data/raw/telecom_churn.csv')
df.drop(columns=['customer_id'], inplace=True, errors='ignore')
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df['total_charges'] = df['total_charges'].fillna(df['total_charges'].median())

print("2. Performing Feature Engineering...")
df["tenure_bucket"] = pd.cut(df["tenure_months"], bins=[0, 12, 24, 48, 72, 120], labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr", "6yr+"])
df["complaint_intensity"] = df["num_complaints_3m"] + df["num_complaints_12m"]
df["engagement_score"] = df["app_logins_30d"] + df["selfcare_transactions_30d"]
df["bill_shock"] = np.where(df["monthly_charges"] > df["monthly_charges"].quantile(0.75), 1, 0)
df["high_value_flag"] = np.where(df["arpu"] > df["arpu"].quantile(0.80), 1, 0)

print("3. Selecting Features via Correlation...")
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
churn_corr = corr_matrix['is_churn'].sort_values(ascending=False)

THRESHOLD = 0.05
selected_num_features = churn_corr[abs(churn_corr) > THRESHOLD].index.tolist()
if 'is_churn' in selected_num_features:
    selected_num_features.remove('is_churn')

selected_cat_features = [
    'gender', 'region_circle', 'connection_type', 'plan_type', 
    'contract_type', 'base_plan_category', 'segment_value', 'tenure_bucket'
]

selected_columns = selected_num_features + selected_cat_features + ['is_churn']
df_model = df[selected_columns]

print("4. Building Preprocessing & Modeling Pipeline...")
X = df_model.drop("is_churn", axis=1)
y = df_model["is_churn"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), selected_num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), selected_cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Calculate the ratio of Non-Churners to Churners for scale_pos_weight
# This ensures XGBoost treats the minority class (Churners) with high priority
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("5. Training the Advanced XGBoost Model...")
# <-- UPGRADED MODEL STEP -->
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=200,          # More trees for better learning
        learning_rate=0.05,        # Slower learning rate prevents overfitting
        max_depth=4,               # Shallower trees generalize better
        scale_pos_weight=pos_weight, # Handles the class imbalance
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)

print("6. Evaluating Model Performance...")
probs = model_pipeline.predict_proba(X_test)[:, 1]

# Adjusted threshold based on XGBoost's better probability calibration
threshold = 0.45 
preds = (probs >= threshold).astype(int)

print("\n==================================================")
print("             XGBOOST CHURN PREDICTION REPORT      ")
print("==================================================")
print(classification_report(y_test, preds))
print("AUC-ROC Score:", round(roc_auc_score(y_test, probs), 4))
print("==================================================\n")

print("7. Saving the Model...")
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "connecttel_churn_pipeline.pkl")
joblib.dump(model_pipeline, model_path)
print(f"✅ Success! XGBoost Model saved in: {os.path.abspath(model_path)}")

1. Loading Data...
2. Performing Feature Engineering...
3. Selecting Features via Correlation...
4. Building Preprocessing & Modeling Pipeline...
5. Training the Advanced XGBoost Model...
6. Evaluating Model Performance...

             XGBOOST CHURN PREDICTION REPORT      
              precision    recall  f1-score   support

           0       0.70      0.52      0.60      2929
           1       0.50      0.68      0.58      2071

    accuracy                           0.59      5000
   macro avg       0.60      0.60      0.59      5000
weighted avg       0.62      0.59      0.59      5000

AUC-ROC Score: 0.6542

7. Saving the Model...
✅ Success! XGBoost Model saved in: d:\C\Churn_Prediction_ConnectTel\models\connecttel_churn_pipeline.pkl
